Steps:
- links
    - get groq to generate N interesting topics 
    - generate wikipedia urls 
- get wiki page content data
- save as txt files

### Generate Wiki Links via Groq

In [2]:
import os
api_key = os.getenv('GROQ_API_KEY')

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

In [4]:
chat = ChatGroq(temperature=0, groq_api_key=api_key, model_name="llama-3.3-70b-versatile")

In [5]:
prompt = """
Generate a list of 10 wikipedia urls for interesting topics related to science and modern technology.
Topics should have short page content.
Output format: 
<list>
[
  "https://en.wikipedia.org/wiki/[topic-1]",
  "https://en.wikipedia.org/wiki/[topic-2]",
...
]
</list>
"""

response = chat.invoke(prompt)
print(response.content)

<list>
[
  "https://en.wikipedia.org/wiki/FAST_Radio_Telescope",
  "https://en.wikipedia.org/wiki/Quantum_entanglement_swapping",
  "https://en.wikipedia.org/wiki/Memristor",
  "https://en.wikipedia.org/wiki/Nanocellulose",
  "https://en.wikipedia.org/wiki/Synthetic_neurobiology",
  "https://en.wikipedia.org/wiki/Graphene_nanosheet",
  "https://en.wikipedia.org/wiki/Spintronics",
  "https://en.wikipedia.org/wiki/Neuromorphic_engineering",
  "https://en.wikipedia.org/wiki/Metasurface",
  "https://en.wikipedia.org/wiki/Topological_quantum_computer"
]
</list>


In [6]:
urls = response.content.strip("<list>").strip("</list>").strip()
type(urls)

str

In [7]:
urls = eval(urls)
type(urls)

list

In [8]:
urls

['https://en.wikipedia.org/wiki/FAST_Radio_Telescope',
 'https://en.wikipedia.org/wiki/Quantum_entanglement_swapping',
 'https://en.wikipedia.org/wiki/Memristor',
 'https://en.wikipedia.org/wiki/Nanocellulose',
 'https://en.wikipedia.org/wiki/Synthetic_neurobiology',
 'https://en.wikipedia.org/wiki/Graphene_nanosheet',
 'https://en.wikipedia.org/wiki/Spintronics',
 'https://en.wikipedia.org/wiki/Neuromorphic_engineering',
 'https://en.wikipedia.org/wiki/Metasurface',
 'https://en.wikipedia.org/wiki/Topological_quantum_computer']

### Get URL Content

In [9]:
from langchain_community.document_loaders import UnstructuredURLLoader

C:\Users\DHRUV\AppData\Local\Temp\ipykernel_11448\2420766737.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [10]:
loader = UnstructuredURLLoader(urls=urls)
data = loader.load()

In [11]:
print(data[1].page_content)

Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.


### Save as .txt

In [12]:
type(data[0].page_content)

str

In [13]:
doc_word_count = [len(doc.page_content.split(" ")) for doc in data]
doc_word_count

[13, 13, 13, 13, 13, 13, 13, 13, 13, 13]

In [14]:
# threshold each document to 4000 words
smaller_docs = [" ".join(doc.page_content.split(" ")[:3000]) for doc in data]
small_doc_word_count = [len(doc.split(" ")) for doc in smaller_docs]
small_doc_word_count

[13, 13, 13, 13, 13, 13, 13, 13, 13, 13]

In [15]:
file_names = [url.split("https://en.wikipedia.org/wiki/")[1] for url in urls]

In [16]:
# create directory
dir_name = "data"
os.makedirs(dir_name, exist_ok=True)

# save each string as a .txt file
for i, text in enumerate(smaller_docs):
    with open(f"{dir_name}/{file_names[i]}.txt", "w") as f:
        f.write(text)